# Metrics for Text Evaluation

* High diagnostic usefulness. Domain-specific semantic:
   * Bio_BERTScore_F1 (Bio-Clinical Balanced Score);
   * Bio_BERTScore_R (Bio-Clinical Recall);
   * Bio_BERTScore_P (Bio-Clinical Precision);
   * Bio_Sentence_CosSim (Biomedical Sentence Embedding Similarity);
* Moderate diagnostic usefulness. Cross-domain semantic:
   * BLEURT (Learned Token-to-Sentence Metric);
   * BERTScore_F1 (General-Domain Contextual Similarity);
   * Semantic_CosSim (General-Domain Sentence Similarity);
* Low diagnostic usefulness:
   * METEOR (Advanced Lexical Overlap);
   * ROUGE-L (Longest Common Subsequence Lexical Overlap);
   * BLEU-4 (4-Gram Lexical Precision).


In [ ]:
import pandas as pd
import nltk
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util
import evaluate
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning)

# ==========================================================
# НАСТРОЙКИ (Задайте здесь имена ваших колонок)
# ==========================================================
TARGET_COLUMN = 'actual_text'  # Имя колонки с истинными значениями
# Список колонок с текстами, сгенерированными разными моделями
#MODEL_COLUMNS = ['test_base_qwen.csv', 'test_finetuned_qwen_v01.csv', 'test_finetuned_qwen_v02.csv', 'test_finetuned_qwen_v03.csv', 'test_finetuned_qwen_v04.csv', 'test_finetuned_qwen_v05.csv', 'test_vit-gpt2_transformer_v01.csv', 'test_vit-gpt2_transformer_v02.csv', 'test_vit-gpt2_transformer_v03.csv', 'test_vit-gpt2_transformer_v04.csv', 'test_vit-gpt2_transformer_v05.csv', 'test_vit-gpt2_transformer_v06.csv', 'test_vit-gpt2_transformer_v07.csv']
MODEL_COLUMNS =['predicted_text']
LANGUAGE = 'en'  # 'ru' для русского, 'en' для английского
# ==========================================================

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def load_data(file_path, target_col, model_cols):
    """Загрузка данных. Ожидается, что в файле есть заголовки (первая строка)."""
    try:
        df = pd.read_csv(file_path)
    except Exception:
        df = pd.read_table(file_path)

    # Проверка наличия всех требуемых колонок
    required_cols = [target_col] + model_cols
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"В файле отсутствуют следующие колонки: {missing_cols}. Доступные: {list(df.columns)}")

    df = df.dropna(subset=required_cols)
    for col in required_cols:
        df[col] = df[col].astype(str).str.strip()

    return df

def calculate_all_metrics(df, target_col, model_cols, lang='ru'):
    """Расчет всех метрик для списка моделей и возврат сводной таблицы."""
    print(f"Начало оценки для моделей: {', '.join(model_cols)}\n")

    # Инициализация словаря для сбора результатов
    results = {'Model': model_cols}

    # 1. N-gram метрики (построчный расчет)
    print("1. Вычисление n-gram метрик (BLEU, METEOR, ROUGE)...")
    bleu_scores = {model: [] for model in model_cols}
    meteor_scores = {model: [] for model in model_cols}
    rouge_scores = {model: [] for model in model_cols}

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smoothie = SmoothingFunction().method4

    for _, row in df.iterrows():
        true_text = str(row[target_col]).lower()
        true_tokens = true_text.split() if lang == 'ru' else nltk.word_tokenize(true_text)

        for model in model_cols:
            gen_text = str(row[model]).lower()
            gen_tokens = gen_text.split() if lang == 'ru' else nltk.word_tokenize(gen_text)

            bleu_scores[model].append(sentence_bleu([true_tokens], gen_tokens, smoothing_function=smoothie))
            meteor_scores[model].append(meteor_score([true_tokens], gen_tokens))

            rouge = scorer.score(str(row[target_col]), str(row[model]))
            rouge_scores[model].append(rouge['rougeL'].fmeasure)

    results['BLEU-4'] = [np.mean(bleu_scores[m]) for m in model_cols]
    results['METEOR'] = [np.mean(meteor_scores[m]) for m in model_cols]
    results['ROUGE-L'] = [np.mean(rouge_scores[m]) for m in model_cols]

    # 2. Семантические метрики (Загружаем модели ОДИН РАЗ для экономии памяти)
    print("2. Загрузка и вычисление семантических метрик (BERTScore, CosSim)...")

    # BERTScore через evaluate (без KeyError)
    bertscore_metric = evaluate.load("bertscore")
    bs_model_type = "xlm-roberta-large" if lang == 'ru' else "roberta-large"

    # Sentence-BERT
    sbert_name = 'cointegrated/LaBSE-en-ru' if lang == 'ru' else 'pritamdeka/S-PubMedBert-MS-MARCO'
    sbert_model = SentenceTransformer(sbert_name)

    # Эмбеддинги эталона считаем один раз
    true_embeddings = sbert_model.encode(
        df[target_col].tolist(),
        convert_to_tensor=True,
        show_progress_bar=False
    )

    bertscore_f1 = []
    cos_sim_scores = []

    for model in model_cols:
        print(f"  -> Оценка модели: {model}")
        gen_texts = df[model].tolist()
        ref_texts = df[target_col].tolist()

        # BERTScore
        bs_results = bertscore_metric.compute(
            predictions=gen_texts,
            references=ref_texts,
            lang=lang,
            model_type=bs_model_type,
            verbose=False
        )
        bertscore_f1.append(np.mean(bs_results['f1']))

        # Cosine Similarity
        gen_embeddings = sbert_model.encode(gen_texts, convert_to_tensor=True, show_progress_bar=False)
        cos_sims = util.cos_sim(true_embeddings, gen_embeddings).diagonal().cpu().numpy()
        cos_sim_scores.append(np.mean(cos_sims))

    results['BERTScore_F1'] = bertscore_f1
    results['Semantic_CosSim'] = cos_sim_scores

    # 3. BLEURT (Опционально, может быть медленно)
    print("3. Вычисление BLEURT...")
    # bleurt_scores = []
    # try:
    #     bleurt = evaluate.load("bleurt", module_type="metric")
    #     for model in model_cols:
    #         scores = bleurt.compute(predictions=df[model].tolist(), references=df[target_col].tolist())
    #         bleurt_scores.append(np.mean(scores['scores']))
    # except Exception as e:
    # print(f"  Пропуск BLEURT: {e}")
    bleurt_scores = [np.nan] * len(model_cols)

    results['BLEURT'] = bleurt_scores

    # Формируем итоговую таблицу
    summary_df = pd.DataFrame(results).set_index('Model')
    return summary_df

def main(file_path):
    print(f"Загрузка данных из {file_path}...")
    df = load_data(file_path, TARGET_COLUMN, MODEL_COLUMNS)
    print(f"Успешно загружено {len(df)} строк.\n")

    summary_df = calculate_all_metrics(df, TARGET_COLUMN, MODEL_COLUMNS, lang=LANGUAGE)

    # Красивый вывод таблицы
    print("\n" + "="*70)
    print("СВОДНАЯ ТАБЛИЦА СРАВНЕНИЯ КАЧЕСТВА ГЕНЕРАЦИИ МОДЕЛЕЙ")
    print("="*70)

    # Настройка отображения pandas для красивого вывода
    pd.set_option('display.float_format', lambda x: '%.4f' % x)
    print(summary_df.to_string())
    print("="*70)

    # Сохранение таблицы в CSV для дальнейшего использования
    output_csv = file_path.replace('.csv', '_metrics_summary.csv').replace('.tsv', '_metrics_summary.csv')
    summary_df.to_csv(output_csv)
    print(f"\nСводная таблица сохранена в: {output_csv}")
    print("Оценка завершена!")

if __name__ == "__main__":
    #FILE_PATH = "/home/kobold/kosareva/kosareva_files/sql/test_all.csv"
    FILE_PATH = "/home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.4.csv"
    # Создаем демо-файл с НЕСКОЛЬКИМИ колонками моделей, если файла нет
    if not os.path.exists(FILE_PATH):
        print(f"Файл {FILE_PATH} не найден. Создаю демо-файл с несколькими моделями...")
        os.makedirs(os.path.dirname(FILE_PATH), exist_ok=True)

        demo_data = {
            'actual_text': [
                "нет признаков острой патологии, органы малого таза без особенностей",
                "признаков перелома не выявлено, костная структура сохранена",
                "выявлена инфильтрация в нижней доле правого легкого, подозрение на пневмонию"
            ],
            'qwen_base': [
                "острая патология не визуализируется, органы малого таза в норме",
                "переломы отсутствуют, целостность костной ткани не нарушена",
                "обнаружен инфильтрат в правом легком, вероятна пневмония"
            ],
            'qwen_instruct': [ # Более качественная генерация для сравнения
                "признаков острой патологии нет, органы малого таза без особенностей",
                "костная структура сохранена, признаков перелома не обнаружено",
                "в нижней доле правого легкого определяется инфильтрация, что соответствует пневмонии"
            ],
            'llama_3_med': [ # Менее качественная генерация для сравнения
                "все нормально, таз в порядке",
                "кости целые",
                "пневмония в легком"
            ]
        }
        pd.DataFrame(demo_data).to_csv(FILE_PATH, index=False, encoding='utf-8')
        print(f"Демо-файл создан. Запуск оценки...\n")

    main(FILE_PATH)

Загрузка данных из /home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.4.csv...
Успешно загружено 4660 строк.

Начало оценки для моделей: predicted_text

1. Вычисление n-gram метрик (BLEU, METEOR, ROUGE)...
2. Загрузка и вычисление семантических метрик (BERTScore, CosSim)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  -> Оценка модели: predicted_text


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


3. Вычисление BLEURT...

СВОДНАЯ ТАБЛИЦА СРАВНЕНИЯ КАЧЕСТВА ГЕНЕРАЦИИ МОДЕЛЕЙ
                BLEU-4  METEOR  ROUGE-L  BERTScore_F1  Semantic_CosSim  BLEURT
Model                                                                         
predicted_text  0.1238  0.3060   0.2935        0.8889           0.9478     NaN

Сводная таблица сохранена в: /home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.4_metrics_summary.csv
Оценка завершена!


In [ ]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import CrossEncoder
import warnings

warnings.filterwarnings("ignore")

# ==========================================================
# НАСТРОЙКИ
# ==========================================================
TARGET_COLUMN = 'actual_text'
# MODEL_COLUMNS = [
#     'test_base_qwen.csv', 'test_finetuned_qwen_v01.csv', 'test_finetuned_qwen_v02.csv',
#     'test_finetuned_qwen_v03.csv', 'test_finetuned_qwen_v04.csv', 'test_finetuned_qwen_v05.csv',
#     'test_vit-gpt2_transformer_v01.csv', 'test_vit-gpt2_transformer_v02.csv',
#     'test_vit-gpt2_transformer_v03.csv', 'test_vit-gpt2_transformer_v04.csv',
#     'test_vit-gpt2_transformer_v05.csv', 'test_vit-gpt2_transformer_v06.csv',
#     'test_vit-gpt2_transformer_v07.csv'
# ]
MODEL_COLUMNS =['predicted_text']
LANGUAGE = 'en'  # 'ru' или 'en'
#FILE_PATH = "/home/kobold/kosareva/kosareva_files/sql/test_all.csv"
FILE_PATH ="/home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.1.csv"
# Рабочие модели Cross-Encoder (проверены на доступность в 2025)
CROSS_ENCODER_MODELS = {
    'en': 'cross-encoder/stsb-roberta-large',        # Английский: STS-задача, отлично для семантической схожести
    'ru': 'DiTy/cross-encoder-russian-mtl',          # Русский: мультизадачная модель
    # Альтернативы, если основные не загрузятся:
    'en_fallback': 'cross-encoder/quora-roberta-base',
    'ru_fallback': 'cointegrated/rubert-tiny2-cased-nli'
}

# Хотите ли вы пытаться загрузить BLEURT (может упасть из-за CUDA)
TRY_BLEURT = False  # Поставьте True, если уверены в совместимости CUDA
# ==========================================================


def load_data(file_path, target_col, model_cols):
    try:
        df = pd.read_csv(file_path)
    except Exception:
        df = pd.read_table(file_path)


def calculate_semantic_scores(df, target_col, model_cols, lang='en'):
    """Основной метод оценки через Cross-Encoder."""
    print(f"Загрузка Cross-Encoder для языка: {lang.upper()}")

    # Выбираем модель по языку
    model_name = CROSS_ENCODER_MODELS.get(lang, CROSS_ENCODER_MODELS['en'])
    fallback_name = CROSS_ENCODER_MODELS.get(f'{lang}_fallback')

    cross_encoder = None
    for name in [model_name, fallback_name]:
        if name is None:
            continue
        try:
            print(f"  -> Попытка загрузить: {name}")
            cross_encoder = CrossEncoder(name)
            print(f"✅ Модель {name} успешно загружена.\n")
            break
        except Exception as e:
            print(f"  ⚠️ Не удалось загрузить {name}: {type(e).__name__}: {str(e)[:150]}")

    if cross_encoder is None:
        print("❌ Ни одна Cross-Encoder модель не загрузилась. Проверьте интернет и доступ к HuggingFace.")
        return pd.DataFrame({'Model': model_cols, 'Semantic_Score': [np.nan] * len(model_cols)}).set_index('Model')

    # Расчет оценок для каждой модели
    results = {'Model': model_cols}
    scores_list = []

    for model in model_cols:
        print(f"  -> Оценка модели: {model}")
        sentence_pairs = list(zip(df[target_col].tolist(), df[model].tolist()))

        # Модель выдает логиты; для STS-моделей это уже значения в диапазоне ~[-1, 5]
        # Превращаем их в диапазон [0, 1] через сигмоиду для единообразия
        raw_scores = cross_encoder.predict(sentence_pairs, show_progress_bar=True, batch_size=32)
        probabilities = 1 / (1 + np.exp(-raw_scores))

        scores_list.append(np.mean(probabilities))

    results['Semantic_Score'] = scores_list
    print("✅ Оценка через Cross-Encoder завершена.\n")

    return pd.DataFrame(results).set_index('Model')


def try_bleurt(df, target_col, model_cols):
    """Опциональная попытка загрузить BLEURT (может упасть из-за CUDA)."""
    if not TRY_BLEURT:
        return None

    print("Попытка загрузки BLEURT (может занять время и упасть из-за CUDA)...")
    try:
        import evaluate
        bleurt = evaluate.load("bleurt", checkpoint="bleurt-20")
        scores_list = []
        for model in model_cols:
            scores = bleurt.compute(
                predictions=df[model].tolist(),
                references=df[target_col].tolist(),
                batch_size=8
            )
            scores_list.append(np.mean(scores['scores']))
        print("✅ BLEURT отработал успешно.\n")
        return scores_list
    except Exception as e:
        print(f"⚠️ BLEURT не сработал: {type(e).__name__}: {str(e)[:200]}")
        print("   Это нормально  CUDA/TF несовместимость. Используем только Cross-Encoder.\n")
        return None


def main():
    print(f"Загрузка данных из {FILE_PATH}...")
    if not os.path.exists(FILE_PATH):
        print(f"❌ Файл не найден: {FILE_PATH}")
        return

    df = load_data(FILE_PATH, TARGET_COLUMN, MODEL_COLUMNS)
    #rint(f"✅ Загружено {len(df)} строк для {len(MODEL_COLUMNS)} моделей.\n")

    # Основной метод: Cross-Encoder
    summary_df = calculate_semantic_scores(df, TARGET_COLUMN, MODEL_COLUMNS, lang=LANGUAGE)

    # Опционально: BLEURT
    bleurt_scores = try_bleurt(df, TARGET_COLUMN, MODEL_COLUMNS)
    if bleurt_scores is not None:
        summary_df['BLEURT_Score'] = bleurt_scores

    # Вывод
    print("\n" + "=" * 80)
    print("РЕЗУЛЬТАТЫ ОЦЕНКИ СМЫСЛОВОЙ СХОЖЕСТИ")
    print("=" * 80)
    pd.set_option('display.float_format', lambda x: '%.4f' % x)
    pd.set_option('display.max_rows', None)
    print(summary_df.to_string())
    print("=" * 80)

    # Сохранение
    output_csv = FILE_PATH.replace('.csv', '_semantic_scores.csv').replace('.tsv', '_semantic_scores.csv')
    summary_df.to_csv(output_csv)
    print(f"\n💾 Результаты сохранены в: {output_csv}")


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import os
import evaluate
from sentence_transformers import SentenceTransformer, util
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# ==========================================================
# 1. НАСТРОЙКИ (Измените под ваши данные)
# ==========================================================
FILE_PATH = "/home/kobold/kosareva/kosareva_files/sql/test_all.csv"
FILE_PATH = "/home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5.csv"
TARGET_COLUMN = 'actual_text'
#MODEL_COLUMNS = ['test_base_qwen.csv', 'test_finetuned_qwen_v01.csv', 'test_finetuned_qwen_v02.csv', 'test_finetuned_qwen_v03.csv', 'test_finetuned_qwen_v04.csv', 'test_finetuned_qwen_v05.csv', 'test_vit-gpt2_transformer_v01.csv', 'test_vit-gpt2_transformer_v02.csv', 'test_vit-gpt2_transformer_v03.csv', 'test_vit-gpt2_transformer_v04.csv', 'test_vit-gpt2_transformer_v05.csv', 'test_vit-gpt2_transformer_v06.csv', 'test_vit-gpt2_transformer_v07.csv']
MODEL_COLUMNS =['predicted_text']

# Выберите язык ваших данных: 'ru' (русский) или 'en' (английский)
LANGUAGE = 'en'

# Настройка моделей в зависимости от языка
if LANGUAGE == 'ru':
    # Для русского языка: специализированная медицинская модель и надежная мультиязычная для BERTScore
    EMBEDDING_MODEL = "blinoff/medru-bert"  # Отличная русская медицинская модель для эмбеддингов
    BERTSCORE_MODEL = "xlm-roberta-large"   # Самая стабильная мультиязычная модель для BERTScore
else:
    # Для английского языка: классические Bio-BERT модели
    EMBEDDING_MODEL = "emilyalsentzer/Bio_ClinicalBERT" # Или "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
    BERTSCORE_MODEL = "roberta-large"       # Стандарт для английского BERTScore

# ==========================================================

def load_data(file_path, target_col, model_cols):
    """Загрузка и валидация данных."""
    try:
        df = pd.read_csv(file_path)
    except Exception:
        df = pd.read_table(file_path)

    required_cols = [target_col] + model_cols
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"В файле нет колонок: {missing}. Доступные: {list(df.columns)}")

    df = df.dropna(subset=required_cols)
    for col in required_cols:
        df[col] = df[col].astype(str).str.strip()
    return df

def calculate_bio_bert_metrics(df, target_col, model_cols, lang='ru'):
    """Расчет метрик с использованием медицинских BERT-моделей."""
    print(f"Запуск оценки через медицинские BERT-модели (Язык: {lang.upper()})")
    print(f"Модель эмбеддингов: {EMBEDDING_MODEL}")
    print(f"Модель BERTScore: {BERTSCORE_MODEL}\n")

    results = {'Model': model_cols}

    # ---------------------------------------------------------
    # ШАГ 1: BERTScore (Precision, Recall, F1)
    # ---------------------------------------------------------
    print("1. Вычисление BERTScore (может занять время)...")
    try:
        # Используем evaluate.load, так как он НЕ выдает KeyError при передаче кастомных путей к моделям
        bertscore_metric = evaluate.load("bertscore")

        bert_p, bert_r, bert_f1 = [], [], []

        for model in model_cols:
            print(f"  -> Обработка модели: {model}")
            scores = bertscore_metric.compute(
                predictions=df[model].tolist(),
                references=df[target_col].tolist(),
                lang="xx" if lang == 'ru' else "en", # "xx" для мультиязычных моделей
                model_type=BERTSCORE_MODEL,
                verbose=False
            )
            bert_p.append(np.mean(scores['precision']))
            bert_r.append(np.mean(scores['recall']))
            bert_f1.append(np.mean(scores['f1']))

        results['Bio_BERTScore_P'] = bert_p
        results['Bio_BERTScore_R'] = bert_r
        results['Bio_BERTScore_F1'] = bert_f1

    except Exception as e:
        print(f"  ⚠️ Ошибка при расчете BERTScore: {e}")
        results['Bio_BERTScore_F1'] = [np.nan] * len(model_cols)

    # ---------------------------------------------------------
    # ШАГ 2: Cosine Similarity (Sentence-BERT)
    # ---------------------------------------------------------
    print("\n2. Вычисление Cosine Similarity (загрузка модели эмбеддингов)...")
    try:
        # Загружаем модель ОДИН раз для всех колонок (экономия памяти и времени)
        bio_sbert = SentenceTransformer(EMBEDDING_MODEL)

        # Эмбеддинги эталона считаем один раз
        print("  -> Кодирование эталонных текстов...")
        target_embeddings = bio_sbert.encode(
            df[target_col].tolist(),
            convert_to_tensor=True,
            show_progress_bar=False
        )

        cos_sim_scores = []
        for model in model_cols:
            print(f"  -> Кодирование и сравнение для модели: {model}")
            gen_embeddings = bio_sbert.encode(
                df[model].tolist(),
                convert_to_tensor=True,
                show_progress_bar=False
            )

            # Косинусная схожесть между векторами эталона и генерации
            similarities = util.cos_sim(target_embeddings, gen_embeddings).diagonal().cpu().numpy()
            cos_sim_scores.append(np.mean(similarities))

        results['Bio_Sentence_CosSim'] = cos_sim_scores

    except Exception as e:
        print(f"  ⚠️ Ошибка при расчете Cosine Similarity: {e}")
        results['Bio_Sentence_CosSim'] = [np.nan] * len(model_cols)

    # Формируем итоговую таблицу
    summary_df = pd.DataFrame(results).set_index('Model')
    return summary_df

def main():
    if not os.path.exists(FILE_PATH):
        print(f"❌ Файл не найден: {FILE_PATH}")
        print("Создаю демо-файл для проверки работы кода...")
        os.makedirs(os.path.dirname(FILE_PATH), exist_ok=True)

        demo_data = {
            'actual_text': [
                "Признаков перелома не выявлено, костная структура сохранена",
                "Выявлена инфильтрация в нижней доле правого легкого, подозрение на пневмонию"
            ],
            'qwen_base': [
                "Переломы отсутствуют, целостность костной ткани не нарушена",
                "Обнаружен инфильтрат в правом легком, вероятна пневмония"
            ],
            'qwen_instruct': [
                "Костная структура сохранена, признаков перелома не обнаружено",
                "В нижней доле правого легкого определяется инфильтрация, что соответствует пневмонии"
            ]
        }
        pd.DataFrame(demo_data).to_csv(FILE_PATH, index=False, encoding='utf-8')
        print("✅ Демо-файл создан.\n")

    print(f"Загрузка данных из {FILE_PATH}...")
    df = load_data(FILE_PATH, TARGET_COLUMN, MODEL_COLUMNS)
    print(f"✅ Успешно загружено {len(df)} строк.\n")

    summary_df = calculate_bio_bert_metrics(df, TARGET_COLUMN, MODEL_COLUMNS, lang=LANGUAGE)

    # Красивый вывод
    print("\n" + "="*80)
    print("СВОДНАЯ ТАБЛИЦА: ОЦЕНКА ЧЕРЕЗ МЕДИЦИНСКИЕ BERT-МОДЕЛИ (Bio-BERT аналог)")
    print("="*80)
    pd.set_option('display.float_format', lambda x: '%.4f' % x)
    print(summary_df.to_string())
    print("="*80)

    # Сохранение
    output_csv = FILE_PATH.replace('.csv', '_bio_bert_metrics.csv').replace('.tsv', '_bio_bert_metrics.csv')
    summary_df.to_csv(output_csv)
    print(f"\n💾 Результаты сохранены в: {output_csv}")
    print("Готово!")

if __name__ == "__main__":
    main()

Загрузка данных из /home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5.csv...
✅ Успешно загружено 4660 строк.

Запуск оценки через медицинские BERT-модели (Язык: EN)
Модель эмбеддингов: emilyalsentzer/Bio_ClinicalBERT
Модель BERTScore: roberta-large

1. Вычисление BERTScore (может занять время)...
  -> Обработка модели: predicted_text


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



2. Вычисление Cosine Similarity (загрузка модели эмбеддингов)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  -> Кодирование эталонных текстов...
  -> Кодирование и сравнение для модели: predicted_text

СВОДНАЯ ТАБЛИЦА: ОЦЕНКА ЧЕРЕЗ МЕДИЦИНСКИЕ BERT-МОДЕЛИ (Bio-BERT аналог)
                Bio_BERTScore_P  Bio_BERTScore_R  Bio_BERTScore_F1  Bio_Sentence_CosSim
Model                                                                                  
predicted_text           0.8922           0.8855            0.8886               0.9494

💾 Результаты сохранены в: /home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5_bio_bert_metrics.csv
Готово!


In [ ]:
import pandas as pd
import numpy as np
import os
from sentence_transformers import CrossEncoder
import warnings

warnings.filterwarnings("ignore")

# ==========================================================
# НАСТРОЙКИ
# ==========================================================
TARGET_COLUMN = 'actual_text'
MODEL_COLUMNS = ['predicted_text'] # Убедитесь, что такой столбец точно есть в вашем CSV
LANGUAGE = 'en'  # 'ru' или 'en'
FILE_PATH = "/home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5.csv"

# Рабочие модели Cross-Encoder
CROSS_ENCODER_MODELS = {
    'en': 'cross-encoder/stsb-roberta-large',
    'ru': 'DiTy/cross-encoder-russian-mtl',
    'en_fallback': 'cross-encoder/quora-roberta-base',
    'ru_fallback': 'cointegrated/rubert-tiny2-cased-nli'
}

TRY_BLEURT = False
# =========================================================


def load_data(file_path, target_col, model_cols):
    try:
        df = pd.read_csv(file_path)
    except Exception:
        df = pd.read_table(file_path)

    # ВАЖНО: Возвращаем DataFrame, иначе функция вернет None
    return df


def calculate_semantic_scores(df, target_col, model_cols, lang='en'):
    """Основной метод оценки через Cross-Encoder."""
    print(f"Загрузка Cross-Encoder для языка: {lang.upper()}")

    model_name = CROSS_ENCODER_MODELS.get(lang, CROSS_ENCODER_MODELS['en'])
    fallback_name = CROSS_ENCODER_MODELS.get(f'{lang}_fallback')

    cross_encoder = None
    for name in [model_name, fallback_name]:
        if name is None:
            continue
        try:
            print(f"  -> Попытка загрузить: {name}")
            cross_encoder = CrossEncoder(name)
            print(f"✅ Модель {name} успешно загружена.\n")
            break
        except Exception as e:
            print(f"  ⚠️ Не удалось загрузить {name}: {type(e).__name__}: {str(e)[:150]}")

    if cross_encoder is None:
        print("❌ Ни одна Cross-Encoder модель не загрузилась. Проверьте интернет и доступ к HuggingFace.")
        return pd.DataFrame({'Model': model_cols, 'Semantic_Score': [np.nan] * len(model_cols)}).set_index('Model')

    results = {'Model': model_cols}
    scores_list = []

    for model in model_cols:
        print(f"  -> Оценка модели: {model}")
        # Проверяем, существуют ли колонки, чтобы избежать новых ошибок
        if target_col not in df.columns or model not in df.columns:
            print(f"  ⚠️ Колонки '{target_col}' или '{model}' не найдены в файле. Пропускаем.")
            scores_list.append(np.nan)
            continue

        sentence_pairs = list(zip(df[target_col].astype(str).tolist(), df[model].astype(str).tolist()))

        raw_scores = cross_encoder.predict(sentence_pairs, show_progress_bar=True, batch_size=32)
        probabilities = 1 / (1 + np.exp(-raw_scores))

        scores_list.append(np.mean(probabilities))

    results['Semantic_Score'] = scores_list
    print("✅ Оценка через Cross-Encoder завершена.\n")

    return pd.DataFrame(results).set_index('Model')


def try_bleurt(df, target_col, model_cols):
    """Опциональная попытка загрузить BLEURT."""
    if not TRY_BLEURT:
        return None

    print("Попытка загрузки BLEURT (может занять время и упасть из-за CUDA)...")
    try:
        import evaluate
        bleurt = evaluate.load("bleurt", checkpoint="bleurt-20")
        scores_list = []
        for model in model_cols:
            scores = bleurt.compute(
                predictions=df[model].astype(str).tolist(),
                references=df[target_col].astype(str).tolist(),
                batch_size=8
            )
            scores_list.append(np.mean(scores['scores']))
        print("✅ BLEURT отработал успешно.\n")
        return scores_list
    except Exception as e:
        print(f"⚠️ BLEURT не сработал: {type(e).__name__}: {str(e)[:200]}")
        print("   Это нормально: CUDA/TF несовместимость. Используем только Cross-Encoder.\n")
        return None


def main():
    print(f"Загрузка данных из {FILE_PATH}...")
    if not os.path.exists(FILE_PATH):
        print(f"❌ Файл не найден: {FILE_PATH}")
        return

    df = load_data(FILE_PATH, TARGET_COLUMN, MODEL_COLUMNS)

    if df is None or df.empty:
        print("❌ Не удалось загрузить данные или файл пуст.")
        return

    print(f"✅ Загружено {len(df)} строк. Найдены колонки: {list(df.columns)}\n")

    # Основной метод: Cross-Encoder
    summary_df = calculate_semantic_scores(df, TARGET_COLUMN, MODEL_COLUMNS, lang=LANGUAGE)

    # Опционально: BLEURT
    bleurt_scores = try_bleurt(df, TARGET_COLUMN, MODEL_COLUMNS)
    if bleurt_scores is not None:
        summary_df['BLEURT_Score'] = bleurt_scores

    # Вывод
    print("\n" + "=" * 80)
    print("РЕЗУЛЬТАТЫ ОЦЕНКИ СМЫСЛОВОЙ СХОЖЕСТИ")
    print("=" * 80)
    pd.set_option('display.float_format', lambda x: '%.4f' % x)
    pd.set_option('display.max_rows', None)
    print(summary_df.to_string())
    print("=" * 80)

    # Сохранение
    output_csv = FILE_PATH.replace('.csv', '_semantic_scores.csv').replace('.tsv', '_semantic_scores.csv')
    summary_df.to_csv(output_csv)
    print(f"\n💾 Результаты сохранены в: {output_csv}")


if __name__ == "__main__":
    main()

Загрузка данных из /home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5.csv...
✅ Загружено 4660 строк. Найдены колонки: ['filename', 'actual_text', 'predicted_text']

Загрузка Cross-Encoder для языка: EN
  -> Попытка загрузить: cross-encoder/stsb-roberta-large


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Модель cross-encoder/stsb-roberta-large успешно загружена.

  -> Оценка модели: predicted_text


Batches:   0%|          | 0/146 [00:00<?, ?it/s]

✅ Оценка через Cross-Encoder завершена.


РЕЗУЛЬТАТЫ ОЦЕНКИ СМЫСЛОВОЙ СХОЖЕСТИ
                Semantic_Score
Model                         
predicted_text          0.6282

💾 Результаты сохранены в: /home/kobold/Downloads/best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5_semantic_scores.csv
